<a href="https://colab.research.google.com/github/RohanYashraj/ifoa-workshop/blob/main/notebooks_v3/01_genai_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · GenAI basics

**Workshop:** AI for Actuaries — From Foundations to AI Agents
**Session / Part:** S2.P1 (reasoner)  ·  **Slides:** S1.P2.18–19
**Author:** Dr Rohan Yashraj Gupta (FIA, FIAI), with Satya Sai Mudigonda
**Date:** 24 July 2026 · Four Points by Sheraton, Whitefield, Bangalore
**Model:** `gemini-3.1-flash-lite` (pinned)  ·  **License:** CC BY-NC 4.0

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rohanyashraj/ifoa-workshop/blob/main/notebooks_v3/01_genai_basics.ipynb)

## What this notebook does
Your first calls to the reasoner: a pinned Gemini call, the CCCE prompting discipline, structured JSON output, and a live look at hallucination.

*All data is hypothetical — ABC Insurer is a fictional entity for teaching only.
The story: Priya Nair (pricing, ABC General) must explain the price of policy
**ABC-MOT-047231** — a 7-year-old SUV, Tier-2, 35% NCB — so her chief actuary
**Arjun Mehta** can sign it.*

## 1. Install & authenticate
Store your key in Colab Secrets (🔑) as `GOOGLE_API_KEY`.

In [14]:
%pip install -q google-genai

In [15]:
import os
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    assert "GOOGLE_API_KEY" in os.environ, "Set GOOGLE_API_KEY (Colab Secrets or env)."

from google import genai
client = genai.Client()          # reads GOOGLE_API_KEY
MODEL = "gemini-3.1-flash-lite"  # PINNED — never 'latest'
print("Reasoner ready:", MODEL)

Reasoner ready: gemini-3.1-flash-lite


## 2. Your first call
The exact call an agent makes under the hood.

In [16]:
from IPython.display import display, Markdown

resp = client.models.generate_content(
    model=MODEL,
    contents="Define IBNR for a board member in one sentence.",
)
display(Markdown(resp.text))

IBNR (Incurred But Not Reported) refers to a reserve fund set aside to cover claims that have already occurred but have not yet been reported to the company, ensuring the organization remains financially solvent for these inevitable future liabilities.

## 3. CCCE — vague vs disciplined
Same request, two specifications.

In [20]:
from IPython.display import display, Markdown

vague = "write about IBNR for our board"

ccce = """Task: write a 2-sentence note for our board on IBNR.
Context: motor book, FY2024, reserve strengthened by Rs 42 crore.
Constraints: use ONLY the figure above, invent no numbers, under 80 words.
Example tone: 'Reserves rose Rs X because ...'."""

for label, prompt in [("BEFORE (vague)", vague), ("AFTER (CCCE)", ccce)]:
    print("=" * 8, label, "=" * 8)
    resp = client.models.generate_content(model=MODEL, contents=prompt)
    display(Markdown(resp.text))
    print("\n")

======== BEFORE (vague) ========


This briefing note is designed for a Board of Directors. It balances technical accuracy with the strategic implications of IBNR (Incurred But Not Reported) reserves, focusing on financial stability, risk management, and regulatory compliance.

***

# Board Briefing: Understanding IBNR (Incurred But Not Reported) Reserves

## 1. Executive Summary
In our financial statements, "IBNR" represents a significant liability. It is the estimate of claims that have already occurred but have not yet been reported to or processed by the company. Because these claims are "hidden" until they surface, they represent a core area of actuarial estimation. Properly managing IBNR is critical to maintaining solvency, meeting regulatory capital requirements, and ensuring the accuracy of our reported earnings.

## 2. What is IBNR?
To understand IBNR, it is helpful to categorize claims into two buckets:
*   **Case Reserves:** Claims that have been reported to us, where we have assessed the likely cost.
*   **IBNR:** The estimated cost of claims that have occurred but are currently invisible to our system. This includes:
    *   **Pure IBNR:** Claims that have happened, but the policyholder has not yet filed a notice.
    *   **IBER (Incurred But Not Enough Reported):** Claims where we know about the event, but our initial estimate is lower than the eventual final settlement cost (development).

## 3. Why IBNR Matters to the Board
As stewards of the company’s capital and reputation, the Board should view IBNR through three lenses:

### A. Profitability and Volatility
Because IBNR is an estimate, it is subject to "reserve development." If we underestimate IBNR, we eventually have to "strengthen reserves," which creates a charge against future earnings. If we overestimate, we may eventually release those reserves, boosting profit—but potentially masking underlying performance issues. Consistent, disciplined reserving is a sign of operational maturity.

### B. Solvency and Capital Adequacy
Regulators (and rating agencies) view IBNR as a primary indicator of financial health. If our reserves are inadequate, we are undercapitalized. This puts the company at risk of regulatory intervention and threatens our ability to pay claims during periods of high frequency or severity (e.g., a catastrophe or a surge in litigation).

### C. Strategic Decision-Making
Pricing and product strategy rely on accurate claims data. If our IBNR models are off, our "Loss Ratio" is wrong. If the Loss Ratio is wrong, we may be underpricing our products, inadvertently growing our market share in unprofitable segments.

## 4. How We Manage the Risk (The Control Framework)
We do not "guess" IBNR. Our approach relies on three pillars:

1.  **Actuarial Rigor:** We utilize professional actuaries who apply industry-standard methodologies (e.g., Chain-Ladder, Bornhuetter-Ferguson). These methods analyze historical patterns—how quickly claims typically emerge and how they typically grow over time.
2.  **Internal Audit and Oversight:** Our reserving process is subject to internal controls. The Audit Committee reviews the assumptions and the methodology used to derive these figures to ensure no "smoothing" of earnings is occurring.
3.  **External Validation:** We engage external actuaries to provide an independent opinion. This "second look" provides the Board with assurance that our reserves fall within a reasonable range of professional practice.

## 5. Key Questions for the Board to Ask
To exercise effective oversight, the Board may consider asking management the following:

*   **"What is the 'actuarial range' for our reserves?"** (Always ask for the range, not just a single number; it shows the uncertainty in the estimate.)
*   **"How have our reserves developed over the last 3–5 years?"** (Are we consistently redundant or deficient? A pattern of deficiency is a red flag.)
*   **"Are there any external factors—such as social inflation, changing legal environments, or economic shifts—that are making our historical patterns less predictive of the future?"**
*   **"Do our current reserve levels provide us with the necessary capital buffer to handle a 'black swan' event?"**

## 6. Conclusion
IBNR is not merely an accounting entry; it is a strategic estimate of our future obligations. By maintaining a robust, transparent, and conservative reserving philosophy, we protect our policyholders, satisfy our regulators, and provide the Board with a true picture of the company’s financial trajectory.

***

*Disclaimer: This document is for informational purposes and should be tailored to the specific industry (e.g., P&C Insurance vs. Health Insurance) and the specific risk appetite of your organization.*



======== AFTER (CCCE) ========


For FY2024, the motor book reserve was strengthened by Rs 42 crore to align with updated actuarial projections. This adjustment ensures our IBNR provisions appropriately reflect recent claims development trends.

## 4. Structured output — agents speak JSON
Guaranteed-parseable output for tools.

In [18]:
from pydantic import BaseModel
from IPython.display import display, Markdown

class Factor(BaseModel):
    name: str
    relativity: float
    direction: str

resp = client.models.generate_content(
    model=MODEL,
    contents="Extract the NCB factor for policy ABC-MOT-047231 (35% NCB, relativity 0.82).",
    config={"response_mime_type": "application/json", "response_schema": Factor},
)

# While JSON is usually plain text, we can wrap it in markdown code blocks for better visibility
display(Markdown(f"```json\n{resp.text}\n```"))

```json
{
  "name": "ABC-MOT-047231",
  "relativity": 0.82,
  "direction": "35% NCB"
}
```

## 5. Hallucination, on cue
Ask for a factor that does not exist. Watch the confident fabrication — this is the *air-filter discount* the guardrail kills in notebook 04.

In [19]:
from IPython.display import display, Markdown

bad = ("What is the exact 'air-filter discount' relativity in the IRDAI motor "
       "tariff for a 7-year-old SUV? Give a number.")

resp = client.models.generate_content(model=MODEL, contents=bad)

display(Markdown(resp.text))
print("\n>>> There is no such factor. The model invented one — plausibly, with a number.")

The answer is **0%**.

There is no "air-filter discount" in the IRDAI motor tariff. In fact, there is no such category or standardized discount for specific mechanical parts (like air filters, oil filters, or spark plugs) in the current IRDAI-regulated motor insurance framework.

Here is the breakdown of why this is the case:

1.  **Tariff De-regulation:** Since the implementation of the **All India Motor Tariff (2002)**, premium rates for "Own Damage" (OD) covers are determined by individual insurers based on their internal underwriting guidelines, not by IRDAI. 
2.  **Parts Coverage:** In a standard comprehensive motor policy, parts are categorized as either **Rubber/Plastic/Metal/Glass/Fiber** for the purpose of "Depreciation" calculation during a claim. An air filter is typically treated as a consumable or a standard spare part; it does not receive a specific "filter discount."
3.  **Depreciation for a 7-year-old vehicle:** Because your vehicle is 7 years old, the insurance company will apply a mandatory depreciation rate on parts replaced during a claim. Per the IRDAI guidelines, the depreciation for parts (other than rubber/nylon/plastic) at the 7-year mark is **40%**. 

**Important Note:** If you have been offered an "air-filter discount" by a garage or a specific agent, it is likely a **promotional marketing tactic** or a private discount being applied to the *retail price of the part* itself during a service, rather than a regulatory discount mandated by the IRDAI motor tariff. Insurance companies do not provide discounts on specific replaceable engine components.


>>> There is no such factor. The model invented one — plausibly, with a number.


## Wrap-up
You can now: call a pinned model, prompt with CCCE, force JSON, and spot a hallucination.

**Next:** `02_models_as_tools.ipynb` — the models an agent calls.

*Demonstrated: the LLM is powerful and confidently wrong — it needs tools that know.*